# AGAR-RL V11 — engine profile, GPU profile, training

This notebook benchmarks the active Colab runtime, initializes V11 from **policy weights only** in the latest valid V10 MaskablePPO checkpoint, and trains toward 20,000,000 V11 timesteps. V11 optimizer, timestep counter, VecNormalize statistics, and opponent league start fresh. If a V11 manifest already exists, the notebook resumes that V11 run instead.

The benchmark reports engine ticks/s, real PPO environment steps/s, rollout geometry, and GPU memory. A100/L4 performance is measured in the runtime where training runs; it is not inferred from a fixed profile. Checkpoints and concise metrics sync to a separate V11 Drive folder. Interrupting the training cell sends SIGINT to the trainer, which saves and syncs before exiting.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO = '/content/agario'
if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', 'clone', 'https://github.com/Albin0903/agario.git', REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', 'checkout', '-B', 'main', 'origin/main'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Repository synced and dependencies installed.')


## Runtime check and engine profile
The engine profile is CPU-side and separate from neural-network training. It includes multiple cells/player and reports the timed phase coverage so raw engine throughput is not confused with SB3 FPS.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab (L4 or A100) before running V11 training.')
torch.set_num_threads(1)
gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024**3)
cpu_count = os.cpu_count() or 1
print(f'GPU: {gpu} | VRAM: {vram_gib:.1f} GiB | logical CPU cores: {cpu_count}')
engine_profile = subprocess.run([
    sys.executable, '-m', 'src.analysis.profile_engine',
    '--steps', '4000', '--warmup-steps', '250', '--json'
], check=True, text=True, capture_output=True)
ENGINE_PROFILE = json.loads(engine_profile.stdout.strip().splitlines()[-1])
print(f"Engine: {ENGINE_PROFILE['measured_engine_steps_per_second']:.1f} ticks/s; "
      f"phase coverage {ENGINE_PROFILE['phase_coverage_percent']:.1f}%")
for phase, percent in ENGINE_PROFILE['phase_percent'].items():
    print(f'{phase:24s} {percent:6.2f}%')
env_profile = subprocess.run([
    sys.executable, '-m', 'src.analysis.profile_env', '--steps', '1000',
    '--warmup-steps', '100', '--json'
], check=True, text=True, capture_output=True)
ENV_PROFILE = json.loads(env_profile.stdout.strip().splitlines()[-1])
print(f"Single AgarEnv: {ENV_PROFILE['environment_steps_per_second']:.1f} decisions/s "
      f"({ENV_PROFILE['physics_ticks_per_second']:.1f} engine ticks/s) with bots + observation")


## V10 source or V11 resume
On the first V11 run, choose the valid V10 MaskablePPO archive with the highest **internal** timestep counter. This skips stale `ppo_final.zip` files whose filename or manifest claims a later step than the counter stored in the archive. For later notebook sessions, the V11 manifest controls resume geometry and the trainer restores the V11 optimizer and normalization state.

In [ ]:
import glob, json, os, re, zipfile
from pathlib import Path

DRIVE_V10 = Path('/content/drive/MyDrive/agario_rl_backup_v10')
DRIVE_V11 = Path('/content/drive/MyDrive/agario_rl_backup_v11')
DRIVE_V11.mkdir(parents=True, exist_ok=True)
V11_MANIFEST = DRIVE_V11 / 'v11_manifest.json'

def stored_timesteps(path):
    try:
        with zipfile.ZipFile(path) as zf:
            data = zf.read('data').decode('utf-8', errors='replace')
        match = re.search(r'\"num_timesteps\"\s*:\s*(\d+)', data)
        return int(match.group(1)) if match else -1
    except (OSError, KeyError, zipfile.BadZipFile):
        return -1

V11_RESUME = V11_MANIFEST.is_file() and (DRIVE_V11 / 'ppo_latest.zip').is_file()
V10_CHECKPOINT = None
if V11_RESUME:
    manifest = json.loads(V11_MANIFEST.read_text(encoding='utf-8'))
    if manifest.get('version') != 'v11':
        raise RuntimeError('Refusing to resume a non-V11 checkpoint from the V11 folder.')
    PROFILE = {k: int(manifest[k]) for k in ('n_envs', 'n_steps', 'batch_size', 'n_epochs')}
    print(f"Resume V11 at {manifest['timesteps']:,} / 20,000,000 steps; saved profile: {PROFILE}")
else:
    if not DRIVE_V10.is_dir():
        raise FileNotFoundError(f'V10 Drive folder not found: {DRIVE_V10}')
    archives = [p for p in DRIVE_V10.glob('*.zip') if p.stat().st_size > 1024]
    candidates = [(stored_timesteps(p), p) for p in archives]
    candidates = [(step, p) for step, p in candidates if step >= 0]
    if not candidates:
        raise FileNotFoundError('No readable V10 checkpoint with a stored PPO timestep was found.')
    v10_step, V10_CHECKPOINT = max(candidates, key=lambda item: item[0])
    print(f'V11 weights source: {V10_CHECKPOINT} (checkpoint counter {v10_step:,})')
    if 'v10_manifest.json' in [p.name for p in DRIVE_V10.iterdir()]:
        v10_manifest = json.loads((DRIVE_V10 / 'v10_manifest.json').read_text(encoding='utf-8'))
        manifest_step = int(v10_manifest.get('timesteps', v10_step))
        if manifest_step != v10_step:
            print(f'V10 manifest counter ({manifest_step:,}) differs; using the archive counter ({v10_step:,}).')


## Choose a measured PPO profile
Fresh V11 runs benchmark short, independent PPO runs from the same V10 policy weights. The profile keeps the optimizer settings fixed and compares CPU worker count, batch size, real end-to-end SB3 FPS, and peak GPU memory. The candidate worker count is capped at the CPU logical-core count, following SB3's guidance for compute-bound environments.

In [ ]:
import json, re, subprocess, sys
from pathlib import Path

if not V11_RESUME:
    worker_candidates = sorted({max(1, min(n, cpu_count)) for n in (4, 8, 12, 16)})
    profile_batch = 2048 if vram_gib >= 32 else (1024 if vram_gib >= 16 else 512)
    candidates = [(n, min(profile_batch, (2048*n))) for n in worker_candidates]
    profile_root = Path('/content/v11_profiles')
    profile_results = []
    for n_envs, batch in candidates:
        run_dir = profile_root / f'envs_{n_envs}_batch_{batch}'
        cmd = [sys.executable, '-m', 'src.training.train_v11',
               '--v10-checkpoint', str(V10_CHECKPOINT), '--profile-run',
               '--n-envs', str(n_envs), '--n-steps', '2048',
               '--batch-size', str(batch), '--n-epochs', '8',
               '--total-timesteps', '20000000', '--save-dir', str(run_dir / 'ppo'),
               '--history-dir', str(run_dir / 'pool'), '--device', 'cuda']
        result = subprocess.run(cmd, text=True, capture_output=True)
        print(result.stdout)
        if result.returncode:
            print(result.stderr[-3000:])
            continue
        match = re.search(r'PROFILE_RESULT steps_per_second=([0-9.]+) max_vram_mb=([0-9.]+)', result.stdout)
        if match:
            profile_results.append({'n_envs': n_envs, 'n_steps': 2048, 'batch_size': batch,
                                   'n_epochs': 8, 'steps_per_second': float(match.group(1)),
                                   'max_vram_mb': float(match.group(2))})
    if not profile_results:
        raise RuntimeError('All GPU profile runs failed; see the printed errors above.')
    PROFILE = max(profile_results, key=lambda row: row['steps_per_second'])
    (DRIVE_V11 / 'v11_profile_results.json').write_text(json.dumps({
        'gpu': gpu, 'vram_gib': vram_gib, 'cpu_count': cpu_count,
        'engine_profile': ENGINE_PROFILE, 'candidates': profile_results,
        'selected': PROFILE}, indent=2), encoding='utf-8')
    print('Measured profile selected:', PROFILE)
else:
    profile_results = []
    print('Profile retained from the V11 manifest; no new optimizer profile is needed for resume.')


## Train or resume V11
The target is a cumulative 20,000,000 V11 timesteps. A fresh run imports V10 policy weights; an interrupted V11 run resumes its actual V11 step counter and optimizer. The trainer writes `ppo_step_*.zip`, `ppo_latest.zip`, `vec_normalize.pkl`, `pool_state.json`, `metrics.jsonl`, and the manifest to Drive. Stop this cell with the Colab interrupt button to request a checkpoint and Drive sync.

In [ ]:
import signal, subprocess, sys
from pathlib import Path

common = [sys.executable, '-m', 'src.training.train_v11',
          '--n-envs', str(PROFILE['n_envs']), '--n-steps', str(PROFILE['n_steps']),
          '--batch-size', str(PROFILE['batch_size']), '--n-epochs', str(PROFILE['n_epochs']),
          '--total-timesteps', '20000000', '--save-dir', '/content/checkpoints/v11',
          '--history-dir', '/content/checkpoints/v11/self_play_pool',
          '--backup-dir', str(DRIVE_V11), '--device', 'cuda']
if V11_RESUME:
    cmd = common + ['--resume-v11', 'auto']
else:
    cmd = common + ['--v10-checkpoint', str(V10_CHECKPOINT)]
print('Starting V11. Interrupting this cell forwards SIGINT so the trainer can save and sync.')
proc = subprocess.Popen(cmd)
try:
    exit_code = proc.wait()
except KeyboardInterrupt:
    print('Forwarding stop request to V11 trainer...')
    proc.send_signal(signal.SIGINT)
    exit_code = proc.wait()
if exit_code != 0:
    raise RuntimeError(f'V11 trainer exited with code {exit_code}.')
print('V11 training stopped cleanly or reached its target.')


## Saved artifacts
Open `MyDrive/agario_rl_backup_v11/metrics.jsonl` for compact progress logs and `v11_profile_results.json` for the measured engine/GPU profile. The benchmark is a throughput selection step; reward quality still needs the fixed-seed V11 evaluation notebook.